In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS209_APPLN_IPC,
    TLS224_APPLN_CPC,
)


In [2]:
# TEST is for checking the pipeline, not estimating sector-level statistics.
ENV = "TEST"
patstat = PatstatClient(env=ENV)
db = patstat.orm()
print(f"PATSTAT environment: {ENV}")


PATSTAT environment: TEST


In [3]:
# Preserve the existing search vocabulary until false positives are assessed.
DRONE_KEYWORDS = (
    "unmanned aerial", "unmanned aircraft", "unmanned air vehicle",
    "unpiloted aerial", "unpiloted aircraft", "unpiloted air vehicle",
    "remotely piloted aircraft", "micro air vehicle",
    "quadrotor", "quadrocopter", "quadcopter", "multicopter",
    "multirotor", "multi-rotor", "hexacopter", "octocopter",
    " uav", "uav ", "drone",
)

DRONE_IPC_CODES = ("B64U",)
DRONE_CPC_CODES = ("B64U", "B64C39/024")

VALIDATION_SEED = 42
VALIDATION_SIZE = 100
VALIDATION_FILE = Path("outputs/validation_soroush_test_baseline_v1.csv")


In [4]:
def keyword_condition(terms):
    """Match any term in the title or abstract (baseline substring logic)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{term.lower()}%") for term in terms),
        *(abstract.like(f"%{term.lower()}%") for term in terms),
    )


def symbol_prefix_condition(column, codes):
    """Match CPC/IPC prefixes after removing padding spaces."""
    symbol = func.replace(column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,
    TLS201_APPLN.docdb_family_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date,
    TLS201_APPLN.earliest_filing_year,
    TLS201_APPLN.granted,
)


def query_by_keywords(*columns):
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(DRONE_KEYWORDS))
    )


def query_by_ipc(*columns):
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_IPC_CODES))
    )


def query_by_cpc(*columns):
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    )


In [5]:
q_keywords = query_by_keywords(
    *APPLN_COLUMNS,
    TLS202_APPLN_TITLE.appln_title,
    TLS203_APPLN_ABSTR.appln_abstract,
).distinct()

df_keywords = patstat.df(q_keywords)
print("Keyword applications:", df_keywords["appln_id"].nunique())
display(df_keywords.head(10))


Keyword applications: 2200


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,2755148,37809459,AU,2006284845,A,2006-08-30,2005,Y,Multi-rotor wind turbine supported by continuo...,"Co-axial, multi-rotor wind turbines, producing..."
1,340447919,45379093,CN,201110240635,A,2011-08-19,2011,N,Tower for multi-rotor wind generating system,The invention discloses a tower for a multi-ro...
2,486236638,59057788,US,201715433697,A,2017-02-15,2013,Y,Multicopter-assisted system and method for lau...,The present disclosure provides various embodi...
3,544272682,56081536,US,202017001254,A,2020-08-24,2014,Y,"Systems and methods for detecting, tracking an...",A system for providing integrated detection an...
4,471234385,57217082,MX,2014015190,A,2014-12-10,2014,Y,RETRACTABLE DEVICE FOR TELEMETRICALLY DETERMIN...,The present invention refers to a retractable ...
5,476005433,58190652,US,201615224497,A,2016-07-29,2015,Y,Unmanned aerial vehicles,"Various systems, methods, for unmanned aerial ..."
6,480390348,58284565,US,201615381749,A,2016-12-16,2015,Y,Apparatus and method for surveying premises of...,"Systems, apparatuses and methods are provided ..."
7,504072555,58718037,US,201615775449,A,2016-11-18,2015,Y,Multi-rotor turbine arranged for coordinated r...,A wind turbine system is described comprising ...
8,506401798,58662140,US,201616067855,A,2016-11-02,2015,N,Drones with Self-Generating Function,A drone with its own power generating function...
9,477812085,58518118,JP,2015203586,A,2015-10-15,2015,Y,AERIAL TYPE INSPECTION DEVICE AND INSPECTION M...,PROBLEM TO BE SOLVED: To provide an aerial typ...


In [6]:
q_ipc = query_by_ipc(*APPLN_COLUMNS, TLS209_APPLN_IPC.ipc_class_symbol).distinct()
df_ipc = patstat.df(q_ipc)
print("IPC applications:", df_ipc["appln_id"].nunique())
display(df_ipc.head(10))


IPC applications: 126


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,ipc_class_symbol
0,513740868,48290734,US,201916265314,A,2019-02-01,2011,Y,B64U 10/25
1,513740868,48290734,US,201916265314,A,2019-02-01,2011,Y,B64U 20/10
2,556168280,49580728,US,202117233087,A,2021-04-16,2012,Y,B64U 101/00
3,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 10/13
4,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 80/60
5,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 80/70
6,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 80/86
7,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 101/30
8,445672561,54332274,EP,15782519,A,2015-04-02,2014,Y,B64U 101/30
9,557385898,63761662,US,202017093420,A,2020-11-09,2014,Y,B64U 10/00


In [7]:
q_cpc = query_by_cpc(*APPLN_COLUMNS, TLS224_APPLN_CPC.cpc_class_symbol).distinct()
df_cpc = patstat.df(q_cpc)
print("CPC applications:", df_cpc["appln_id"].nunique())
display(df_cpc.head(10))


CPC applications: 2190


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,cpc_class_symbol
0,52104346,24390749,US,59725632,A,1932-03-07,1932,Y,B64U 10/60
1,54372193,25514354,US,96850378,A,1978-12-11,1978,Y,B64U 10/60
2,46566715,22019089,US,5881587,A,1987-06-05,1987,Y,B64U 70/30
3,50237261,27320303,US,36900589,A,1989-06-20,1988,Y,B64U 10/60
4,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64C 39/024
5,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 10/10
6,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 10/25
7,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/20
8,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/21
9,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/294


In [8]:
application_id = TLS201_APPLN.appln_id.label("appln_id")

drone_ids = (
    query_by_keywords(application_id)
    .union(query_by_ipc(application_id), query_by_cpc(application_id))
    .subquery()
)

q_drone = (
    db.query(*APPLN_COLUMNS, TLS202_APPLN_TITLE.appln_title, TLS203_APPLN_ABSTR.appln_abstract)
    .select_from(TLS201_APPLN)
    .join(drone_ids, TLS201_APPLN.appln_id == drone_ids.c.appln_id)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .distinct()
    .order_by(TLS201_APPLN.appln_id)
)

df_drone = patstat.df(q_drone)
print("Combined distinct applications:", df_drone["appln_id"].nunique())
display(df_drone.head(10))


Combined distinct applications: 3466


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,24746,37809459,EP,06802616,A,2006-08-30,2005,N,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,"Co-axial, multi-rotor wind turbines, producing..."
1,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...
2,352060,33524069,EP,04017945,A,2004-07-29,2003,N,A flying machine,The flying machine (1) comprises a supporting ...
3,2755148,37809459,AU,2006284845,A,2006-08-30,2005,Y,Multi-rotor wind turbine supported by continuo...,"Co-axial, multi-rotor wind turbines, producing..."
4,4305581,25672538,CA,594383,A,1989-03-21,1987,Y,MONITORING SYSTEM FOR POWER LINES AND RIGHT-OF...,Application Of: Roosevelt A. Fernandes For: Mo...
5,5009380,37809459,CA,2620922,A,2006-08-30,2005,N,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,"Co-axial, multi-rotor wind turbines, producing..."
6,6750614,26877844,CN,01807474,A,2001-02-12,2000,N,Remotely piloted aircraft,None
7,7611249,5221267,CN,98111274,A,1998-05-04,1998,N,Wind motor,"The present invention relates to a wind motor,..."
8,13609024,7942894,DE,20010668,U,2000-06-15,2000,Y,Windenergieanlage Multi-Rotor-Prinzip,None
9,14034049,11074674,DE,60119462,T,2001-09-10,2000,Y,Unbemannte mobile Vorrichtung,None


In [9]:
coverage = pd.Series({
    "Keywords": df_keywords["appln_id"].nunique(),
    "IPC": df_ipc["appln_id"].nunique(),
    "CPC": df_cpc["appln_id"].nunique(),
    "Combined union": df_drone["appln_id"].nunique(),
}, name="Distinct applications")
display(coverage)

valid_id = df_drone["docdb_family_id"].notna() & df_drone["docdb_family_id"].gt(0)
years = pd.to_numeric(df_drone["earliest_filing_year"], errors="coerce")

print("Rows:", len(df_drone))
print("Distinct families:", df_drone.loc[valid_id, "docdb_family_id"].nunique())
print("Applications missing a valid family:", df_drone.loc[~valid_id, "appln_id"].nunique())
print("Applications with unavailable earliest filing year:",
      df_drone.loc[years.isna() | years.eq(9999), "appln_id"].nunique())


Keywords          2200
IPC                126
CPC               2190
Combined union    3466
Name: Distinct applications, dtype: int64

Rows: 3466
Distinct families: 2469
Applications missing a valid family: 0
Applications with unavailable earliest filing year: 0


In [10]:
df_diagnostics = df_drone.copy()
for name, data in (("keywords", df_keywords), ("ipc", df_ipc), ("cpc", df_cpc)):
    df_diagnostics[f"found_by_{name}"] = df_diagnostics["appln_id"].isin(data["appln_id"])


def retrieval_label(row):
    labels = [name.upper() if name != "keywords" else "Keywords"
              for name in ("keywords", "ipc", "cpc") if row[f"found_by_{name}"]]
    return " + ".join(labels)


df_diagnostics["retrieval_strategy"] = df_diagnostics.apply(retrieval_label, axis=1)

strategy_counts = (
    df_diagnostics.groupby("retrieval_strategy")["appln_id"]
    .nunique().sort_values(ascending=False)
)
display(strategy_counts.rename("Distinct applications"))

keyword_only = df_diagnostics.loc[
    df_diagnostics["found_by_keywords"] &
    ~df_diagnostics["found_by_ipc"] &
    ~df_diagnostics["found_by_cpc"]
].copy()
print("Keyword-only applications:", keyword_only["appln_id"].nunique())


retrieval_strategy
Keywords                1243
CPC                     1213
Keywords + CPC           884
Keywords + IPC + CPC      57
IPC + CPC                 36
IPC                       17
Keywords + IPC            16
Name: Distinct applications, dtype: int64

Keyword-only applications: 1243


In [11]:
def matching_keywords(title, abstract, terms=DRONE_KEYWORDS):
    """Reproduce the baseline substring check on one retrieved record."""
    title_text = str(title) if pd.notna(title) else ""
    abstract_text = str(abstract) if pd.notna(abstract) else ""
    combined = f"{title_text} {abstract_text}".lower()
    return [term for term in terms if term.lower() in combined]


wind_hits = df_diagnostics.loc[
    df_diagnostics["appln_title"].fillna("").str.contains(
        r"wind turbine|wind generating", case=False, regex=True
    )
].copy()

wind_hits["matching_keywords"] = wind_hits.apply(
    lambda row: matching_keywords(row["appln_title"], row["appln_abstract"]),
    axis=1,
)

print("Potential wind-related rows:", len(wind_hits))
display(wind_hits[[
    "appln_id", "docdb_family_id", "appln_title",
    "retrieval_strategy", "matching_keywords"
]].head(20))


Potential wind-related rows: 306


,appln_id,docdb_family_id,appln_title,retrieval_strategy,matching_keywords
0,24746,37809459,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,Keywords,[multi-rotor]
3,2755148,37809459,Multi-rotor wind turbine supported by continuo...,Keywords,[multi-rotor]
5,5009380,37809459,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,Keywords,[multi-rotor]
14,21513231,37605568,Multi-rotor wind turbine,Keywords,[multi-rotor]
39,50267387,37809459,Stationary co-axial multi-rotor wind turbine s...,Keywords,[multi-rotor]
58,53415480,27128626,Side-furling co-axial multi-rotor wind turbine,Keywords,[multi-rotor]
59,53585350,34990046,Multi-rotor wind turbine with generator as cou...,Keywords,[multi-rotor]
72,54481109,27128626,"Balanced, high output, rapid rotation wind tur...",Keywords,[multi-rotor]
74,54613098,37809459,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,Keywords,[multi-rotor]
94,274265725,41529649,Tethered autonomous air vehicle with wind turb...,Keywords + CPC,"[unmanned aerial, uav, uav ]"


In [12]:
def inspect_application(application_id):
    """Display retrieval evidence and full text for one matching application."""
    records = df_diagnostics.loc[df_diagnostics["appln_id"].eq(application_id)]
    if records.empty:
        print("Application not found.")
        return

    ipc = sorted(df_ipc.loc[df_ipc["appln_id"].eq(application_id),
                            "ipc_class_symbol"].dropna().unique().tolist())
    cpc = sorted(df_cpc.loc[df_cpc["appln_id"].eq(application_id),
                            "cpc_class_symbol"].dropna().unique().tolist())

    for _, row in records.iterrows():
        print("Application:", application_id, "| Family:", row["docdb_family_id"])
        print("Retrieved by:", row["retrieval_strategy"])
        print("Matched keywords:", matching_keywords(row["appln_title"], row["appln_abstract"]))
        print("Matched IPC:", ipc, "| Matched CPC:", cpc)
        print("TITLE:", row["appln_title"] if pd.notna(row["appln_title"]) else "[missing]")
        print("ABSTRACT:", row["appln_abstract"] if pd.notna(row["appln_abstract"]) else "[missing]")
        print("-" * 70)


if not wind_hits.empty:
    inspect_application(wind_hits.iloc[0]["appln_id"])


Application: 24746 | Family: 37809459
Retrieved by: Keywords
Matched keywords: ['multi-rotor']
Matched IPC: [] | Matched CPC: []
TITLE: MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUOUS CENTRAL DRIVESHAFT
ABSTRACT: Co-axial, multi-rotor wind turbines, producing more power than a single rotor of the same diameter, are made even more powerful by increasing driveshaft length and supporting the driveshaft from more than one point. Sacrificing the ability to aim, for the extra length to support additional rotors, results in a more powerful co-axial multi-rotor turbine, especially useful for areas with a predominantly unidirectional wind resource. Ideally the turbine is placed at an offset angle a (alpha) from the wind direction, which, in combination with proper spacing between rotors, allows fresh wind to reach each rotor, so that all rotors contribute toward rotation of the driveshaft. Placing the driveshaft under tension can raise critical speeds and reduce the number of intermediate supp

In [13]:
valid_family = df_diagnostics["docdb_family_id"].notna() & df_diagnostics["docdb_family_id"].gt(0)
df_family_hits = df_diagnostics.loc[valid_family].copy()

df_families = (
    df_family_hits.groupby("docdb_family_id", as_index=False)
    .agg(
        n_matched_applications=("appln_id", "nunique"),
        matched_application_ids=("appln_id", lambda ids: sorted(set(ids))),
        found_by_keywords=("found_by_keywords", "any"),
        found_by_ipc=("found_by_ipc", "any"),
        found_by_cpc=("found_by_cpc", "any"),
    )
    .sort_values("docdb_family_id").reset_index(drop=True)
)
df_families["retrieval_strategy"] = df_families.apply(retrieval_label, axis=1)
print("Candidate patent families:", len(df_families))
display(df_families.head(10))


Candidate patent families: 2469


,docdb_family_id,n_matched_applications,matched_application_ids,found_by_keywords,found_by_ipc,found_by_cpc,retrieval_strategy
0,5221267,1,[7611249],True,False,False,Keywords
1,7942894,1,[13609024],True,False,False,Keywords
2,11074674,2,"[14034049, 54342525]",False,False,True,CPC
3,11459538,1,[52322637],False,False,True,CPC
4,19895696,1,[42023759],True,False,False,Keywords
5,19914776,2,"[42123841, 53662805]",False,False,True,CPC
6,22019089,1,[46566715],False,False,True,CPC
7,23142817,1,[49394130],True,False,True,Keywords + CPC
8,23143245,1,[49394876],True,False,True,Keywords + CPC
9,24390749,1,[52104346],False,False,True,CPC


In [14]:
validation_sample = df_families.sample(
    n=min(VALIDATION_SIZE, len(df_families)),
    random_state=VALIDATION_SEED,
).reset_index(drop=True)

validation_sample["relevance"] = ""
validation_sample["reason"] = ""
print("Families selected for review:", len(validation_sample))
display(validation_sample[["docdb_family_id", "retrieval_strategy", "relevance", "reason"]].head(10))


Families selected for review: 100


,docdb_family_id,retrieval_strategy,relevance,reason
0,77478287,Keywords + CPC,,
1,68426213,CPC,,
2,72422945,Keywords + CPC,,
3,72457225,Keywords,,
4,65664234,Keywords + CPC,,
5,59497411,Keywords + CPC,,
6,59144956,Keywords + CPC,,
7,62904216,Keywords + CPC,,
8,56849600,Keywords + CPC,,
9,50708445,Keywords,,


In [15]:
VALIDATION_FILE.parent.mkdir(parents=True, exist_ok=True)
label_columns = ["docdb_family_id", "relevance", "reason"]

if VALIDATION_FILE.exists():
    saved = pd.read_csv(VALIDATION_FILE, keep_default_na=False,
                        dtype={"docdb_family_id": "int64"})
    current_ids = set(validation_sample["docdb_family_id"].astype(int))
    saved_ids = set(saved["docdb_family_id"].astype(int))
    if current_ids != saved_ids or saved["docdb_family_id"].duplicated().any():
        raise ValueError("Existing CSV belongs to another sample. Change VALIDATION_FILE before continuing.")
    saved = saved.set_index("docdb_family_id")
    for column in ("relevance", "reason"):
        validation_sample[column] = (
            validation_sample["docdb_family_id"].map(saved[column]).fillna("")
        )
    print("Restored existing manual assessments from:", VALIDATION_FILE)
else:
    validation_sample[label_columns].to_csv(VALIDATION_FILE, index=False)
    print("Created blank validation template:", VALIDATION_FILE)


def label_family(family_id, relevance, reason):
    """Save a manual decision after reviewing the family."""
    allowed = {"relevant", "not relevant", "uncertain"}
    if relevance not in allowed:
        raise ValueError(f"Relevance must be one of: {sorted(allowed)}")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("Provide a short, evidence-based reason.")
    match = validation_sample["docdb_family_id"].eq(int(family_id))
    if not match.any():
        raise ValueError("Family is not part of the sampled 100 families.")
    validation_sample.loc[match, ["relevance", "reason"]] = [relevance, reason.strip()]
    validation_sample[label_columns].to_csv(VALIDATION_FILE, index=False)
    print("Saved manual label for family:", family_id)


def inspect_family(family_id):
    """Inspect each directly matched application in a DOCDB family."""
    application_ids = (
        df_family_hits.loc[df_family_hits["docdb_family_id"].eq(int(family_id)), "appln_id"]
        .drop_duplicates().sort_values()
    )
    if application_ids.empty:
        print("Family not found.")
        return
    for app_id in application_ids:
        inspect_application(app_id)


print("Next family ID to inspect:",
      int(validation_sample.iloc[0]["docdb_family_id"]) if not validation_sample.empty else "None")


Restored existing manual assessments from: outputs/validation_soroush_test_baseline_v1.csv
Next family ID to inspect: 77478287


In [16]:
# Review a family first. Then, in a separate cell, call label_family(...).
if not validation_sample.empty:
    inspect_family(int(validation_sample.iloc[0]["docdb_family_id"]))

# Example only; DO NOT uncomment without checking the patent family.
# label_family(123456, "relevant", "UAV-specific flight-control invention")


Application: 556553257 | Family: 77478287
Retrieved by: Keywords + CPC
Matched keywords: ['unmanned aerial']
Matched IPC: [] | Matched CPC: ['B64C  39/024', 'B64U2101/00', 'B64U2101/30']
TITLE: Unmanned aerial vehicle bird repelling device
ABSTRACT: The invention discloses an unmanned aerial vehicle bird repelling device, relates to the technical field of unmanned aerial vehicles, and aims to solve the problems of long bird repelling time and slow effect due to the fact that manual patrol and vehicle patrol are adopted in airports generally and multiple persons and multiple devices are allocated for patrol. The unmanned aerial vehicle bird repelling device is characterized in that a second bearing shell body is arranged below a first bearing shell body, a positioning table is arranged above the first bearing shell body, the positioning table and the first bearing shell body are integrally formed, six tweeters are mounted on the two sides of the positioning table, five flashing lights a

In [17]:
labels = validation_sample["relevance"].fillna("").str.strip()
completed = labels.isin(("relevant", "not relevant", "uncertain"))
decided = labels.isin(("relevant", "not relevant"))

print(f"Reviewed: {completed.sum()}/{len(validation_sample)}")
print("Relevant:", labels.eq("relevant").sum())
print("Not relevant:", labels.eq("not relevant").sum())
print("Uncertain:", labels.eq("uncertain").sum())

if completed.all() and decided.all() and len(validation_sample):
    precision = labels.eq("relevant").mean()
    print(f"Estimated precision (complete sample): {precision:.1%}")
else:
    print("Precision not reported: finish all manual labels and resolve uncertain cases.")


Reviewed: 0/100
Relevant: 0
Not relevant: 0
Uncertain: 0
Precision not reported: finish all manual labels and resolve uncertain cases.


In [18]:
def retrieve_family_members():
    """Find every available PATSTAT application in a matched DOCDB family."""
    selected_families = (
        db.query(TLS201_APPLN.docdb_family_id.label("docdb_family_id"))
        .select_from(TLS201_APPLN)
        .join(drone_ids, TLS201_APPLN.appln_id == drone_ids.c.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.isnot(None), TLS201_APPLN.docdb_family_id > 0)
        .distinct().subquery()
    )
    query = (
        db.query(*APPLN_COLUMNS)
        .select_from(TLS201_APPLN)
        .join(selected_families, TLS201_APPLN.docdb_family_id == selected_families.c.docdb_family_id)
        .distinct()
        .order_by(TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id)
    )
    return patstat.df(query)


df_family_members = retrieve_family_members()
print("Directly matched applications:", df_drone["appln_id"].nunique())
print("All retrieved family members:", df_family_members["appln_id"].nunique())
print("Retrieved families:", df_family_members["docdb_family_id"].nunique())
display(df_family_members.head())


Directly matched applications: 3466
All retrieved family members: 3562
Retrieved families: 2469


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted
0,7611249,5221267,CN,98111274,A,1998-05-04,1998,N
1,13609024,7942894,DE,20010668,U,2000-06-15,2000,Y
2,14034049,11074674,DE,60119462,T,2001-09-10,2000,Y
3,54342525,11074674,US,96144701,A,2001-09-25,2000,Y
4,52322637,11459538,US,62669703,A,2003-07-25,2002,Y


In [19]:
selected_family_ids = df_families["docdb_family_id"].dropna().astype(int).unique().tolist()

q_applicants = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS206_PERSON.person_id,
        TLS206_PERSON.person_name,
        TLS206_PERSON.psn_name,
        TLS206_PERSON.person_ctry_code,
    )
    .select_from(TLS201_APPLN)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(
        TLS201_APPLN.docdb_family_id.in_(selected_family_ids),
        TLS207_PERS_APPLN.applt_seq_nr > 0,
    )
    .distinct()
)

df_applicants = patstat.df(q_applicants)
print("Application-applicant rows:", len(df_applicants))
print("Families with applicants:", df_applicants["docdb_family_id"].nunique())
print("Families missing applicants:", len(set(selected_family_ids) - set(df_applicants["docdb_family_id"])))
display(df_applicants.head(10))


Application-applicant rows: 4192
Families with applicants: 2445
Families missing applicants: 24


,docdb_family_id,appln_id,person_id,person_name,psn_name,person_ctry_code
0,24390749,52104346,39274467,RESPESS AERONAUTICAL ENGINEERING CORPORATION,RESPESS AERONAUTICAL ENGINEERING CORPORATION,
1,25514354,54372193,10672277,Loyd; Miles L.,LOYD; MILES L.,US
2,22019089,46566715,5804877,Yoffe; Meir,YOFFE; MEIR,IL
3,25672538,47286207,5713356,Fernandes; Roosevelt A.,FERNANDES; ROOSEVELT A.,US
4,25672538,4305581,16222234,"FERNANDES, ROOSEVELT A.",FERNANDES; ROOSEVELT A.,US
5,27320303,50237261,5262143,Yamaha Hatsudoki Kabushiki Kaisha,YAMAHA MOTOR COMPANY,JP
6,19895696,42023759,33776463,"NORSKE STATS OLJESELSKAP AS, DEN","NORSKE STATS OLJESELSKAP AS, DEN",NO
7,24729242,52709811,6613349,"Advanced Aerospace Technologies, Inc.",ADVANCED AEROSPACE TECHNOLOGIES,US
8,5221267,7611249,18014679,ZHU JIE,ZHU JIE,CN
9,23142817,49394130,5410835,Sikorsky Aircraft Corporation,SIKORSKY AIRCRAFT CORPORATION,US


In [20]:
members = df_family_members.copy()
years = pd.to_numeric(members["earliest_filing_year"], errors="coerce")
members["valid_earliest_filing_year"] = years.where(years.between(1, 9998))

family_years = (
    members.groupby("docdb_family_id", as_index=False)
    .agg(family_earliest_filing_year_proxy=("valid_earliest_filing_year", "min"))
)
family_years["family_earliest_filing_year_proxy"] = (
    family_years["family_earliest_filing_year_proxy"].astype("Int64")
)

df_families = (
    df_families.drop(columns=["family_earliest_filing_year_proxy"], errors="ignore")
    .merge(family_years, on="docdb_family_id", how="left", validate="one_to_one")
)

print("Families without an earliest-filing-year proxy:",
      df_families["family_earliest_filing_year_proxy"].isna().sum())
display(df_families[["docdb_family_id", "family_earliest_filing_year_proxy"]].head(10))


Families without an earliest-filing-year proxy: 0


,docdb_family_id,family_earliest_filing_year_proxy
0,5221267,1998
1,7942894,2000
2,11074674,2000
3,11459538,2002
4,19895696,1992
5,19914776,2003
6,22019089,1987
7,23142817,1999
8,23143245,1999
9,24390749,1932


In [21]:
def normalize_names(series):
    """Harmonize case, whitespace, and Unicode; do not infer corporate ownership."""
    return (
        series.astype("string").str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip().str.upper().replace("", pd.NA)
    )


df_applicants_clean = df_applicants.copy()
df_applicants_clean["person_name_clean"] = normalize_names(df_applicants_clean["person_name"])
df_applicants_clean["psn_name_clean"] = normalize_names(df_applicants_clean["psn_name"])
df_applicants_clean["applicant_name"] = (
    df_applicants_clean["psn_name_clean"].fillna(df_applicants_clean["person_name_clean"])
)
df_applicants_clean["name_source"] = "missing"
df_applicants_clean.loc[df_applicants_clean["person_name_clean"].notna(), "name_source"] = "person_name"
df_applicants_clean.loc[df_applicants_clean["psn_name_clean"].notna(), "name_source"] = "psn_name"

print("Unique normalized applicant names:", df_applicants_clean["applicant_name"].nunique())
print("Missing names:", df_applicants_clean["applicant_name"].isna().sum())
display(df_applicants_clean[["person_name", "psn_name", "applicant_name", "name_source"]].head(10))


Unique normalized applicant names: 2096
Missing names: 0


,person_name,psn_name,applicant_name,name_source
0,RESPESS AERONAUTICAL ENGINEERING CORPORATION,RESPESS AERONAUTICAL ENGINEERING CORPORATION,RESPESS AERONAUTICAL ENGINEERING CORPORATION,psn_name
1,Loyd; Miles L.,LOYD; MILES L.,LOYD; MILES L.,psn_name
2,Yoffe; Meir,YOFFE; MEIR,YOFFE; MEIR,psn_name
3,Fernandes; Roosevelt A.,FERNANDES; ROOSEVELT A.,FERNANDES; ROOSEVELT A.,psn_name
4,"FERNANDES, ROOSEVELT A.",FERNANDES; ROOSEVELT A.,FERNANDES; ROOSEVELT A.,psn_name
5,Yamaha Hatsudoki Kabushiki Kaisha,YAMAHA MOTOR COMPANY,YAMAHA MOTOR COMPANY,psn_name
6,"NORSKE STATS OLJESELSKAP AS, DEN","NORSKE STATS OLJESELSKAP AS, DEN","NORSKE STATS OLJESELSKAP AS, DEN",psn_name
7,"Advanced Aerospace Technologies, Inc.",ADVANCED AEROSPACE TECHNOLOGIES,ADVANCED AEROSPACE TECHNOLOGIES,psn_name
8,ZHU JIE,ZHU JIE,ZHU JIE,psn_name
9,Sikorsky Aircraft Corporation,SIKORSKY AIRCRAFT CORPORATION,SIKORSKY AIRCRAFT CORPORATION,psn_name


In [22]:
print("STEP 0 · DEVELOPMENT CHECKPOINT")
print("Environment:", ENV)
print("Matched applications:", df_drone["appln_id"].nunique())
print("Candidate DOCDB families:", len(df_families))
print("Validation decisions:", completed.sum(), "/", len(validation_sample))
print("Validation CSV:", VALIDATION_FILE)
print("Outstanding: review false positives, finish manual labels, verify priority year,")
print("             agree on 2000–2022/2023 period, validate on the PROD corpus.")


STEP 0 · DEVELOPMENT CHECKPOINT
Environment: TEST
Matched applications: 3466
Candidate DOCDB families: 2469
Validation decisions: 0 / 100
Validation CSV: outputs/validation_soroush_test_baseline_v1.csv
Outstanding: review false positives, finish manual labels, verify priority year,
             agree on 2000–2022/2023 period, validate on the PROD corpus.
